In [1]:
import os
import glob
import pandas as pd
import numpy as np

train_candidates = glob.glob(os.path.join("**", "train.csv"), recursive=True)
if not train_candidates:
    raise FileNotFoundError("train.csv not found in any subdirectory.")
train_path = train_candidates[0]

df_train = pd.read_csv(train_path)

TARGETS = df_train.columns[-6:]
print("Target columns:", list(TARGETS))

ALPHA = 0.2

vote_sums = df_train[TARGETS].sum().astype(float) + ALPHA
total_votes = vote_sums.sum()
global_probs = vote_sums / total_votes
print("Global class probabilities (fallback) with smoothing:")
print(global_probs)

eeg_group = df_train.groupby("eeg_id")[list(TARGETS)].sum()
eeg_counts = eeg_group.sum(axis=1)  # total votes per eeg_id (no smoothing)
eeg_group_smoothed = eeg_group + ALPHA
eeg_totals = eeg_group_smoothed.sum(axis=1).replace(0, np.nan)
per_eeg_probs = eeg_group_smoothed.div(eeg_totals, axis=0).fillna(global_probs)

patient_group = df_train.groupby("patient_id")[list(TARGETS)].sum()
patient_counts = patient_group.sum(axis=1)  # total votes per patient_id (no smoothing)
patient_group_smoothed = patient_group + ALPHA
patient_totals = patient_group_smoothed.sum(axis=1).replace(0, np.nan)
per_patient_probs = patient_group_smoothed.div(patient_totals, axis=0).fillna(
    global_probs
)

eeg_counts_dict = eeg_counts.to_dict()
patient_counts_dict = patient_counts.to_dict()




Target columns: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Global class probabilities (fallback) with smoothing:
seizure_vote    0.120552
lpd_vote        0.157550
gpd_vote        0.171072
lrda_vote       0.133116
grda_vote       0.148106
other_vote      0.269604
dtype: float64


In [2]:
test_candidates = glob.glob(os.path.join("**", "test.csv"), recursive=True)
if not test_candidates:
    raise FileNotFoundError("test.csv not found in any subdirectory.")
test_path = test_candidates[0]

df_test = pd.read_csv(test_path)

submission = pd.DataFrame()
submission["eeg_id"] = df_test["eeg_id"]

# Adjust blending constants to increase global influence and reduce per‑eeg / per‑patient impact
BASE_GLOBAL_WEIGHT = 0.5  # stronger global fallback
EEG_SCALE = 30.0  # larger denominator → smaller per‑eeg weight
PATIENT_SCALE = 30.0  # larger denominator → smaller per‑patient weight
MAX_EEG_WEIGHT = 0.25  # limit per‑eeg contribution
MAX_PAT_WEIGHT = 0.25  # limit per‑patient contribution

for col in TARGETS:
    eeg_vals = df_test["eeg_id"].map(per_eeg_probs[col])
    patient_vals = df_test["patient_id"].map(per_patient_probs[col])

    final_vals = np.full(len(df_test), np.nan)

    for idx, row in df_test.iterrows():
        eid = row["eeg_id"]
        pid = row["patient_id"]

        cnt_eeg = eeg_counts_dict.get(eid, 0.0)
        cnt_pat = patient_counts_dict.get(pid, 0.0)

        w_eeg = min((cnt_eeg / (cnt_eeg + EEG_SCALE)) * MAX_EEG_WEIGHT, MAX_EEG_WEIGHT)
        w_pat = min(
            (cnt_pat / (cnt_pat + PATIENT_SCALE)) * MAX_PAT_WEIGHT, MAX_PAT_WEIGHT
        )

        total = w_eeg + w_pat + BASE_GLOBAL_WEIGHT
        if total > 1.0:
            scale = 1.0 / total
            w_eeg *= scale
            w_pat *= scale
            w_glob = 0.0
        else:
            w_glob = 1.0 - (w_eeg + w_pat)

        prob_eeg = eeg_vals.iloc[idx] if pd.notna(eeg_vals.iloc[idx]) else 0.0
        prob_pat = patient_vals.iloc[idx] if pd.notna(patient_vals.iloc[idx]) else 0.0
        prob_glob = global_probs[col]

        final_vals[idx] = w_glob * prob_glob + w_eeg * prob_eeg + w_pat * prob_pat

    submission[col] = final_vals

row_sums = submission[TARGETS].sum(axis=1)
tolerance = 1e-6
if not np.allclose(row_sums, 1.0, atol=tolerance):
    correction = 1.0 / row_sums
    submission[TARGETS] = submission[TARGETS].multiply(correction, axis=0)

assert np.allclose(
    submission[TARGETS].sum(axis=1), 1.0, atol=1e-6
), "Probabilities do not sum to 1"

output_path = "submission.csv"
submission.to_csv(output_path, index=False)
print(f"Submission written to {output_path}, shape: {submission.shape}")

Submission written to submission.csv, shape: (9850, 7)
